Extraemos archivo jsonl que contiene la data cruda

#**🧪 PRUEBA TÉCNICA – INGENIERO DE DATOS**
**Empresa**: PUNTORED  
**Objetivo**: Calidad y observabilidad.  
**Fecha creación**: 29/03/2025  
**Autor**: Jorge Andres Mosquera  
**Version**: 1.0   
**Fecha modificación**:  
**Motivo modificación**:  

Variables de notepad

In [0]:
catalog_name = "jorge_catalog"
schema_name  = "transacciones"

silver_path = f"/Volumes/{catalog_name}/{schema_name}/silver"
gold_path   = f"/Volumes/{catalog_name}/{schema_name}/gold"

Validaciones de calidad: Conteo de registros

In [0]:
print("Transactions:", spark.read.format("delta").load(silver_path + "/transactions").count())
print("Users:", spark.read.format("delta").load(silver_path + "/users").count())
print("Details:", spark.read.format("delta").load(silver_path + "/transaction_details").count())

Transactions: 50000
Users: 40
Details: 50000


cheque nulls 

In [0]:
from pyspark.sql.functions import col

df_tx = spark.read.format("delta").load(silver_path + "/transactions")

for c in df_tx.columns:
    nulls = df_tx.filter(col(c).isNull()).count()
    print(f"Columna {c}: {nulls} valores nulos")


Columna transaction_id: 0 valores nulos
Columna user_id: 0 valores nulos
Columna amount: 0 valores nulos
Columna status: 0 valores nulos
Columna created_at: 0 valores nulos


ultima fecha actualización

In [0]:
from pyspark.sql.functions import max

df_tx.select(max("created_at")).show()

+-------------------+
|    max(created_at)|
+-------------------+
|2024-09-28 23:54:45|
+-------------------+



Validaciones por pipeline

In [0]:
# amount >= 0
invalid_amounts = df_tx.filter(col("amount") < 0).count()

# status válido
invalid_status = df_tx.filter(~col("status").isin(["approved", "rejected", "pending"])).count()

# transaction_id único
duplicates = df_tx.groupBy("transaction_id").count().filter(col("count") > 1).count()

Guarda datos de validaciion en tabla 

In [0]:
from pyspark.sql import Row

results = [
    Row(validation="amount_non_negative", success=(invalid_amounts == 0)),
    Row(validation="status_valid", success=(invalid_status == 0)),
    Row(validation="transaction_id_unique", success=(duplicates == 0))
]

df_results = spark.createDataFrame(results)

# Guardar como tabla permanente en UC
df_results.write.format("delta").mode("append").saveAsTable("jorge_catalog.transacciones.gold_quality_logs")

In [0]:
%sql
SELECT * FROM jorge_catalog.transacciones.gold_quality_logs;


validation,success
amount_non_negative,true
status_valid,false
transaction_id_unique,true
amount_non_negative,true
status_valid,false
transaction_id_unique,true
amount_non_negative,true
status_valid,false
transaction_id_unique,true


In [0]:
from pyspark.sql import Row

# Contenido del README como filas
readme_rows = [
    Row(section="Bronze", description="Almacena datos crudos (JSONL) tal como llegan."),
    Row(section="Silver", description="Aplica reglas de negocio básicas y normaliza los datos."),
    Row(section="Gold", description="Genera métricas analíticas y vistas para negocio."),
    Row(section="Calidad", description="Validaciones de integridad y observabilidad del pipeline.")
]

df_readme = spark.createDataFrame(readme_rows)

# Guardar como tabla en UC
df_readme.write.format("delta").mode("overwrite").saveAsTable("jorge_catalog.transacciones.readme_pipeline")

print("README guardado en tabla: jorge_catalog.transacciones.readme_pipeline")

README guardado en tabla: jorge_catalog.transacciones.readme_pipeline


In [0]:
%sql
SELECT * FROM jorge_catalog.transacciones.readme_pipeline


section,description
Bronze,Almacena datos crudos (JSONL) tal como llegan.
Silver,Aplica reglas de negocio básicas y normaliza los datos.
Gold,Genera métricas analíticas y vistas para negocio.
Calidad,Validaciones de integridad y observabilidad del pipeline.
